# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd
import numpy as np

In [2]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [3]:
# explorar datasets

display(orders.head())
orders.info()

display(catalog.head())
catalog.info()

display(marketing.head())
marketing.info()


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [4]:
# ============ ORDERS ============
 
# --- Nulos ---
print('Nulos en orders antes:\n', orders.isnull().sum())
 
# Fechas: convertir a datetime
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'], errors='coerce')
 
# Texto: normalizar mayúsculas/minúsculas en categóricas
# (el dataset mezcla 'Mexico'/'mexico', 'Argentina'/'argentina', etc.)

cols_categoricas = ['pais', 'dispositivo', 'fuente_referencia', 'categoria_producto']
for col in cols_categoricas:
    orders[col] = orders[col].str.strip().str.title()
orders['nombre_producto'] = orders['nombre_producto'].str.strip()

# Nulos en categóricas: se documentan, no se inventan valores.
# Para el análisis de KPIs y funnel, se eliminan filas sin país o producto,
# porque son dimensiones clave y no se pueden imputar de forma confiable.
orders = orders.dropna(subset=['pais', 'nombre_producto', 'categoria_producto',
                                'dispositivo', 'fuente_referencia'])
 
# Numéricas: cantidad y precio_unitario nulos -> no se pueden inferir con certeza,
# se eliminan (son pocas filas, <0.5% del total)
orders = orders.dropna(subset=['cantidad', 'precio_unitario', 'monto_descuento'])
 
# --- Valores inválidos ---
# cantidad negativa: pedidos inválidos, se eliminan
orders = orders[orders['cantidad'] > 0]
 
# outliers extremos de cantidad (10,000 / 20,000 unidades): error de carga de datos.
# Se eliminan usando un umbral de negocio razonable (ej. > 50 unidades por pedido no es un pedido normal de e-commerce de consumo).
orders = orders[orders['cantidad'] <= 50]
 
# monto_total ya no debería tener negativos tras filtrar cantidad, se valida:
orders = orders[orders['monto_total'] >= 0]
 
# --- Consistencia de montos ---
# Verificar que monto_total ≈ cantidad * precio_unitario - monto_descuento
orders['monto_calculado'] = orders['cantidad'] * orders['precio_unitario'] - orders['monto_descuento']
inconsistentes = orders[~np.isclose(orders['monto_total'], orders['monto_calculado'], atol=0.05)]
print(f'Filas con monto_total inconsistente: {len(inconsistentes)}')
orders = orders.drop(columns='monto_calculado')
 
# --- Duplicados ---
print('Duplicados en orders:', orders.duplicated().sum())
orders = orders.drop_duplicates()
 
# --- Normalizar categoria_producto para que coincida con catalog ---
# catalog usa "Electrónica" (con tilde) y orders "Electronica" (sin tilde)
orders['categoria_producto'] = orders['categoria_producto'].replace({
    'Electronica': 'Electrónica'
})
 
print('Nulos en orders después:\n', orders.isnull().sum())
print('Shape final orders:', orders.shape)
 


Nulos en orders antes:
 id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     80
cantidad               50
precio_unitario        50
monto_descuento        50
monto_total             0
dtype: int64
Filas con monto_total inconsistente: 0
Duplicados en orders: 100
Nulos en orders después:
 id_pedido             0
id_usuario            0
fecha_hora_pedido     0
pais                  0
dispositivo           0
fuente_referencia     0
nombre_producto       0
categoria_producto    0
cantidad              0
precio_unitario       0
monto_descuento       0
monto_total           0
dtype: int64
Shape final orders: (24590, 12)


In [5]:

# ============ CATALOG ============
print('Nulos en catalog:\n', catalog.isnull().sum())
print('Duplicados en catalog:', catalog.duplicated().sum())
 
catalog['nombre_producto'] = catalog['nombre_producto'].str.strip()
catalog['categoria_producto'] = catalog['categoria_producto'].str.strip()
catalog['proveedor'] = catalog['proveedor'].str.strip()
 
# Costos no pueden ser negativos ni cero
catalog = catalog[catalog['costo_unitario'] > 0]
 
 

Nulos en catalog:
 nombre_producto       0
categoria_producto    0
costo_unitario        0
proveedor             0
dtype: int64
Duplicados en catalog: 0


In [6]:

# ============ MARKETING ============
marketing['fecha'] = pd.to_datetime(marketing['fecha'], errors='coerce')
marketing['pais'] = marketing['pais'].str.strip().str.title()
 
print('Nulos en marketing antes:\n', marketing.isnull().sum())
 
# canal nulo: no se puede inferir el canal real, se marca como 'desconocido'
# en vez de eliminar la fila (el gasto sigue siendo válido para el total de marketing)
marketing['canal'] = marketing['canal'].fillna('desconocido')
 
# gasto no puede ser negativo ni cero
marketing = marketing[marketing['gasto'] > 0]
 
print('Duplicados en marketing:', marketing.duplicated().sum())
marketing = marketing.drop_duplicates()
 
print('Nulos en marketing después:\n', marketing.isnull().sum())
print('Shape final marketing:', marketing.shape)
 
 # Verificación final
# ==========================================

print("ORDERS")
display(orders.info())
display(orders.isna().sum())

print("\nCATALOG")
display(catalog.info())
display(catalog.isna().sum())

print("\nMARKETING")
display(marketing.info())
display(marketing.isna().sum())

Nulos en marketing antes:
 fecha           0
pais            0
id_campaña      0
canal         101
gasto           0
dtype: int64
Duplicados en marketing: 0
Nulos en marketing después:
 fecha         0
pais          0
id_campaña    0
canal         0
gasto         0
dtype: int64
Shape final marketing: (1620, 5)
ORDERS
<class 'pandas.core.frame.DataFrame'>
Int64Index: 24590 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24590 non-null  object        
 1   id_usuario          24590 non-null  object        
 2   fecha_hora_pedido   24590 non-null  datetime64[ns]
 3   pais                24590 non-null  object        
 4   dispositivo         24590 non-null  object        
 5   fuente_referencia   24590 non-null  object        
 6   nombre_producto     24590 non-null  object        
 7   categoria_producto  24590 non-null  object        
 8   cantida

None

id_pedido             0
id_usuario            0
fecha_hora_pedido     0
pais                  0
dispositivo           0
fuente_referencia     0
nombre_producto       0
categoria_producto    0
cantidad              0
precio_unitario       0
monto_descuento       0
monto_total           0
dtype: int64


CATALOG
<class 'pandas.core.frame.DataFrame'>
Int64Index: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 280.0+ bytes


None

nombre_producto       0
categoria_producto    0
costo_unitario        0
proveedor             0
dtype: int64


MARKETING
<class 'pandas.core.frame.DataFrame'>
Int64Index: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1620 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 75.9+ KB


None

fecha         0
pais          0
id_campaña    0
canal         0
gasto         0
dtype: int64

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [7]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

print('Archivos exportados: orders_clean.csv, catalog_clean.csv, marketing_clean.csv')

Archivos exportados: orders_clean.csv, catalog_clean.csv, marketing_clean.csv


---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [8]:

# Parte 1: Rentabilidad del negocio
# ---------------------------------------------
 
# Cruzar orders con catalog para obtener el costo de cada producto
merged = orders.merge(catalog[['nombre_producto', 'costo_unitario']],
                       on='nombre_producto', how='left')
 
print('Productos sin costo tras el merge:', merged['costo_unitario'].isnull().sum())
 
merged['costo_total_pedido'] = merged['cantidad'] * merged['costo_unitario']
 
revenue_total = merged['monto_total'].sum()
costo_total = merged['costo_total_pedido'].sum()
marketing_total = marketing['gasto'].sum()
profit = revenue_total - costo_total - marketing_total
 
print(f'Revenue total:        ${revenue_total:,.2f}')
print(f'Costo total (COGS):   ${costo_total:,.2f}')
print(f'Inversión marketing:  ${marketing_total:,.2f}')
print(f'Profit:               ${profit:,.2f}')
print(f'Margen sobre revenue: {profit/revenue_total*100:.1f}%')
 

Productos sin costo tras el merge: 0
Revenue total:        $9,491,675.14
Costo total (COGS):   $3,783,478.80
Inversión marketing:  $2,871,843.53
Profit:               $2,836,352.81
Margen sobre revenue: 29.9%


In [9]:
# Parte 2: Comportamiento de ventas
# ---------------------------------------------
 
ticket_promedio = merged['monto_total'].mean()
cantidad_promedio = merged['cantidad'].mean()
 
print(f'\nTicket promedio por orden: ${ticket_promedio:,.2f}')
print(f'Cantidad promedio de productos por orden: {cantidad_promedio:.2f}')
 
# Producto más vendido (por unidades)
producto_mas_vendido = merged.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False)
print('\nProducto más vendido por unidades:')
print(producto_mas_vendido)
 
# Producto que más revenue genera (puede diferir del más vendido en unidades)
producto_mayor_revenue = merged.groupby('nombre_producto')['monto_total'].sum().sort_values(ascending=False)
print('\nProducto con mayor revenue:')
print(producto_mayor_revenue)
 
# Gasto en marketing por canal
gasto_por_canal = marketing.groupby('canal')['gasto'].sum().sort_values(ascending=False)
print('\nGasto en marketing por canal:')
print(gasto_por_canal)
 
# Profit por categoría de producto (extra: útil para "segmentos rentables")
merged['profit_pedido'] = merged['monto_total'] - merged['costo_total_pedido']
profit_por_categoria = merged.groupby('categoria_producto')['profit_pedido'].sum().sort_values(ascending=False)
print('\nProfit por categoría de producto:')
print(profit_por_categoria)


Ticket promedio por orden: $386.00
Cantidad promedio de productos por orden: 1.51

Producto más vendido por unidades:
nombre_producto
Vacuum-Pro-Black        6203.0
Jacket-Winter-M         6185.0
Blender-XL-Red          6184.0
Sneakers-Urban-42       6085.0
Laptop-Gaming-16GB      4160.0
Tablet-Standard-64GB    4108.0
Phone-Pro-128GB         4088.0
Name: cantidad, dtype: float64

Producto con mayor revenue:
nombre_producto
Vacuum-Pro-Black        1602954.86
Jacket-Winter-M         1588706.38
Blender-XL-Red          1587286.76
Sneakers-Urban-42       1542350.52
Laptop-Gaming-16GB      1076026.73
Tablet-Standard-64GB    1050399.57
Phone-Pro-128GB         1043950.32
Name: monto_total, dtype: float64

Gasto en marketing por canal:
canal
social         918043.21
organic        913533.01
paid_search    863088.21
desconocido    177179.10
Name: gasto, dtype: float64

Profit por categoría de producto:
categoria_producto
Hogar          1994930.06
Electrónica    1857814.58
Moda           1855451

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [10]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [11]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [12]:
query_eventos_unicos = '''
SELECT nombre_evento, COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY usuarios_unicos DESC;
'''
eventos_unicos = pd.read_sql(query_eventos_unicos, con=engine)
print(eventos_unicos)

      nombre_evento  usuarios_unicos
0       first_visit             7796
1       add_to_cart             7634
2       select_item             7582
3    begin_checkout             7208
4  add_payment_info             6250
5          purchase             6240


In [13]:
# PARTE 1: Totales del funnel
# ======================
query_totals = '''
SELECT
    COUNT(DISTINCT CASE WHEN nombre_evento = 'first_visit' THEN id_usuario END) AS paso_1_visita,
    COUNT(DISTINCT CASE WHEN nombre_evento = 'view_product' THEN id_usuario END) AS paso_2_ver_producto,
    COUNT(DISTINCT CASE WHEN nombre_evento = 'add_to_cart' THEN id_usuario END) AS paso_3_carrito,
    COUNT(DISTINCT CASE WHEN nombre_evento = 'checkout_start' THEN id_usuario END) AS paso_4_checkout,
    COUNT(DISTINCT CASE WHEN nombre_evento = 'purchase' THEN id_usuario END) AS paso_5_compra
FROM events;
'''
totals = pd.read_sql(query_totals, con=engine)
totals


,paso_1_visita,paso_2_ver_producto,paso_3_carrito,paso_4_checkout,paso_5_compra
0,7796,0,7634,0,6240


In [14]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH totales AS (
    SELECT
        COUNT(DISTINCT CASE WHEN nombre_evento = 'first_visit' THEN id_usuario END) AS paso_1,
        COUNT(DISTINCT CASE WHEN nombre_evento = 'view_product' THEN id_usuario END) AS paso_2,
        COUNT(DISTINCT CASE WHEN nombre_evento = 'add_to_cart' THEN id_usuario END) AS paso_3,
        COUNT(DISTINCT CASE WHEN nombre_evento = 'checkout_start' THEN id_usuario END) AS paso_4,
        COUNT(DISTINCT CASE WHEN nombre_evento = 'purchase' THEN id_usuario END) AS paso_5
    FROM events
)
SELECT
    paso_1,
    paso_2,
    paso_3,
    paso_4,
    paso_5,
    ROUND(paso_2::numeric / NULLIF(paso_1,0) * 100, 2) AS conv_1_a_2,
    ROUND(paso_3::numeric / NULLIF(paso_2,0) * 100, 2) AS conv_2_a_3,
    ROUND(paso_4::numeric / NULLIF(paso_3,0) * 100, 2) AS conv_3_a_4,
    ROUND(paso_5::numeric / NULLIF(paso_4,0) * 100, 2) AS conv_4_a_5,
    ROUND(paso_5::numeric / NULLIF(paso_1,0) * 100, 2) AS conversion_final
FROM totales;
'''
conversion = pd.read_sql(query_conversion, con=engine)
conversion
 

,paso_1,paso_2,paso_3,paso_4,paso_5,conv_1_a_2,conv_2_a_3,conv_3_a_4,conv_4_a_5,conversion_final
0,7796,0,7634,0,6240,0.0,None,0.0,None,80.04


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [15]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [16]:
# Explorar tabla user_activity
# =========================

query_user_activity = '''
SELECT * FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)
 

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [17]:
# Retención por cohortes
# ======================

 
query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT
        id_usuario,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS mes_cohorte
    FROM users
),
tamanio_cohorte AS (
    SELECT mes_cohorte, COUNT(DISTINCT id_usuario) AS usuarios_iniciales
    FROM cohortes
    GROUP BY mes_cohorte
),
actividad_semanal AS (
    SELECT
        c.id_usuario,
        c.mes_cohorte,
        CASE
            WHEN ua.dias_despues_registro BETWEEN 1 AND 7 THEN 1
            WHEN ua.dias_despues_registro BETWEEN 8 AND 14 THEN 2
            WHEN ua.dias_despues_registro BETWEEN 15 AND 21 THEN 3
            ELSE NULL
        END AS semana,
        ua.activo
    FROM cohortes c
    JOIN user_activity ua ON c.id_usuario = ua.id_usuario
    WHERE ua.activo = 1
),
retenidos AS (
    SELECT
        mes_cohorte,
        COUNT(DISTINCT CASE WHEN semana = 1 THEN id_usuario END) AS retenido_w1,
        COUNT(DISTINCT CASE WHEN semana = 2 THEN id_usuario END) AS retenido_w2,
        COUNT(DISTINCT CASE WHEN semana = 3 THEN id_usuario END) AS retenido_w3
    FROM actividad_semanal
    GROUP BY mes_cohorte
)
SELECT
    tc.mes_cohorte,
    tc.usuarios_iniciales,
    r.retenido_w1,
    r.retenido_w2,
    r.retenido_w3,
    ROUND(r.retenido_w1::numeric / NULLIF(tc.usuarios_iniciales,0) * 100, 1) AS semana_1,
    ROUND(r.retenido_w2::numeric / NULLIF(tc.usuarios_iniciales,0) * 100, 1) AS semana_2,
    ROUND(r.retenido_w3::numeric / NULLIF(tc.usuarios_iniciales,0) * 100, 1) AS semana_3
FROM tamanio_cohorte tc
LEFT JOIN retenidos r ON tc.mes_cohorte = r.mes_cohorte
ORDER BY tc.mes_cohorte;
'''
 
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final


,mes_cohorte,usuarios_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01 00:00:00+00:00,1627,697,668,656,42.8,41.1,40.3
1,2025-02-01 00:00:00+00:00,1444,611,609,635,42.3,42.2,44.0
2,2025-03-01 00:00:00+00:00,1636,677,705,690,41.4,43.1,42.2
3,2025-04-01 00:00:00+00:00,1606,680,697,663,42.3,43.4,41.3
4,2025-05-01 00:00:00+00:00,1687,695,676,706,41.2,40.1,41.8


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** ...
   - **H₁ (Hipótesis alternativa):** ...
   
**Test estadístico:** ...  
**Nivel de significancia alpha:** ...

In [18]:

import pandas as pd
import numpy as np
from scipy.stats import norm, chi2_contingency
 
exp = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
 
# --- Vista rápida y validación de calidad ---
print(exp.shape)
print(exp.isnull().sum())
print(exp['variante'].value_counts())
print('Duplicados:', exp.duplicated().sum())
 
# --- Tasa de conversión por variante ---
resumen = exp.groupby('variante')['convirtio'].agg(['count', 'sum', 'mean'])
resumen.columns = ['n_usuarios', 'conversiones', 'tasa_conversion']
print(resumen)




(10000, 7)
id_usuario         0
variante           0
convirtio          0
dispositivo        0
pais               0
duracion_sesion    0
timestamp          0
dtype: int64
tratamiento    5035
control        4965
Name: variante, dtype: int64
Duplicados: 0
             n_usuarios  conversiones  tasa_conversion
variante                                              
control            4965           779         0.156898
tratamiento        5035           820         0.162860


In [19]:
control = exp[exp['variante'] == 'control']['convirtio']
tratamiento = exp[exp['variante'] == 'tratamiento']['convirtio']
 
n_control, n_trat = len(control), len(tratamiento)
x_control, x_trat = control.sum(), tratamiento.sum()
p_control, p_trat = x_control / n_control, x_trat / n_trat
 
# proporción combinada bajo H0
p_pool = (x_control + x_trat) / (n_control + n_trat)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_control + 1/n_trat))
z = (p_trat - p_control) / se
p_value = 2 * (1 - norm.cdf(abs(z)))
 
alpha = 0.05
 
print(f'\nTasa control:      {p_control:.4f} ({x_control}/{n_control})')
print(f'Tasa tratamiento:  {p_trat:.4f} ({x_trat}/{n_trat})')
print(f'Diferencia:        {p_trat - p_control:+.4f} ({(p_trat-p_control)/p_control*100:+.2f}% relativo)')
print(f'z-estadístico:     {z:.4f}')
print(f'p-value:           {p_value:.4f}')
 
if p_value < alpha:
    print(f'\np-value < {alpha} -> se rechaza H0: la diferencia es estadísticamente significativa.')
else:
    print(f'\np-value >= {alpha} -> no se rechaza H0: no hay evidencia suficiente de diferencia real.')
 
# --- Validación cruzada con chi-cuadrado (mismo resultado, otro método) ---
tabla_contingencia = pd.crosstab(exp['variante'], exp['convirtio'])
chi2, p_chi2, dof, freq_esperada = chi2_contingency(tabla_contingencia)
print(f'\n[Validación] chi2 = {chi2:.4f}, p-value = {p_chi2:.4f}')
 


Tasa control:      0.1569 (779/4965)
Tasa tratamiento:  0.1629 (820/5035)
Diferencia:        +0.0060 (+3.80% relativo)
z-estadístico:     0.8133
p-value:           0.4161

p-value >= 0.05 -> no se rechaza H0: no hay evidencia suficiente de diferencia real.

[Validación] chi2 = 0.6178, p-value = 0.4319


In [20]:
print('Conclusión: NO se rechaza H0. La diferencia observada es pequeña y consistente con variación aleatoria (ruido muestral), no con un efecto real de la nueva UI de checkout.')
print('Recomendación de negocio: no hay evidencia suficiente para lanzar la nueva UI de checkout a producción basándose solo en este resultado.')
print('Se recomienda NO tomar de decisión definitiva y, si el negocio quiere seguir explorando esta hipótesis, correr el experimento con una muestra más grande o por más tiempo antes de decidir.')

Conclusión: NO se rechaza H0. La diferencia observada es pequeña y consistente con variación aleatoria (ruido muestral), no con un efecto real de la nueva UI de checkout.
Recomendación de negocio: no hay evidencia suficiente para lanzar la nueva UI de checkout a producción basándose solo en este resultado.
Se recomienda NO tomar de decisión definitiva y, si el negocio quiere seguir explorando esta hipótesis, correr el experimento con una muestra más grande o por más tiempo antes de decidir.


---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [ ]:

# https://app.powerbi.com/groups/me/reports/6d5a4c17-ce22-4775-808b-af00dfeb91f3?ctid=3ae3c7c7-ea51-46de-9540-12f4110eac9b&pbi_source=linkShare

# https://drive.google.com/drive/folders/1IvML0SYMXYy5Fln6I817fWpBttHmYOB5?usp=drive_link

# 